# 22 - Custom / Official RAG UI Demo

A lightweight Gradio UI for asking questions over either the official-law index or instructor-provided custom documents. For fast demo use, retrieval-only mode can show sources without loading the LLM.

In [ ]:
!pip install -q -U gradio "sentence-transformers>=3.0.0" transformers accelerate bitsandbytes peft faiss-cpu rank-bm25 pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OFFICIAL_INDEX = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
CUSTOM_INDEX = DRIVE_ROOT / 'indexes/custom_v1'

print('Device:', device)
print('Official index exists:', OFFICIAL_INDEX.exists())
print('Custom index exists:', CUSTOM_INDEX.exists())

In [ ]:
from functools import lru_cache
from typing import Any

import gradio as gr

from src.generation import generate_text, load_llm, retrieve_and_optionally_rerank
from src.prompting import build_rag_prompt
from src.retrieval import RetrievalEngine

DEFAULT_LLM = 'Qwen/Qwen3-32B'

@lru_cache(maxsize=4)
def get_engine(index_root: str):
    return RetrievalEngine(index_root=index_root, device=device)

@lru_cache(maxsize=2)
def get_llm(model_name: str, load_in_4bit: bool):
    return load_llm(model_name=model_name, device=device, load_in_4bit=load_in_4bit)

def index_for_mode(corpus_mode: str) -> Path:
    if corpus_mode == 'custom_docs':
        return CUSTOM_INDEX
    return OFFICIAL_INDEX

def format_sources(items: list[dict[str, Any]]) -> str:
    lines = []
    for i, item in enumerate(items, start=1):
        citation = item.get('citation_label', '')
        key = item.get('article_key', '')
        score = item.get('score', '')
        text = str(item.get('generation_text') or item.get('retrieval_text') or '')
        text = text.replace('\n', ' ')[:900]
        lines.append(f'[{i}] {citation}\nKey: {key} | Score: {score}\n{text}')
    return '\n\n'.join(lines)

def answer_question(
    question: str,
    corpus_mode: str,
    generate_answer: bool,
    llm_model: str,
    top_k_context: int,
    candidate_k: int,
    max_new_tokens: int,
):
    question = (question or '').strip()
    if not question:
        return 'Soru bo? olamaz.', ''
    index_root = index_for_mode(corpus_mode)
    if not index_root.exists():
        return f'Index bulunamad?: {index_root}. ?nce 21_custom_data_pipeline_demo.ipynb veya ilgili index notebookunu ?al??t?r.', ''

    engine = get_engine(str(index_root))
    retrieved = retrieve_and_optionally_rerank(
        engine=engine,
        question=question,
        retriever_mode='dense',
        top_k_context=int(top_k_context),
        candidate_k=int(candidate_k),
        dense_weight=1.0,
        bm25_weight=0.0,
        reranker=None,
    )
    sources = format_sources(retrieved)
    if not generate_answer:
        return 'Yan?t ?retimi kapal?. A?a??da en ilgili kaynaklar listelendi.', sources

    tokenizer, model = get_llm(llm_model.strip() or DEFAULT_LLM, True)
    prompt = build_rag_prompt(question, retrieved, max_context_chars=9000)
    answer = generate_text(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=int(max_new_tokens),
        temperature=0.1,
        top_p=0.9,
        input_max_length=8192,
    )
    return answer, sources

with gr.Blocks(title='Turkish Legal RAG Demo') as demo:
    gr.Markdown('## Turkish Legal RAG Demo')
    gr.Markdown('Official-law veya custom dok?man indexi ?zerinde soru sor. Heavy LLM mode cevap ba??na zaman alabilir.')
    with gr.Row():
        corpus_mode = gr.Dropdown(['official_law', 'custom_docs'], value='official_law', label='Corpus mode')
        generate_answer_box = gr.Checkbox(value=False, label='LLM cevab? ?ret')
    question_box = gr.Textbox(label='Soru', lines=3, placeholder='?rn. Kiraya veren hangi durumlarda tahliye davas? a?abilir?')
    with gr.Row():
        llm_model_box = gr.Textbox(value=DEFAULT_LLM, label='LLM model')
        top_k_box = gr.Slider(1, 10, value=5, step=1, label='Top-k context')
        candidate_k_box = gr.Slider(5, 50, value=30, step=5, label='Candidate-k')
        max_tokens_box = gr.Slider(64, 768, value=384, step=64, label='Max new tokens')
    run_button = gr.Button('Sor')
    answer_box = gr.Textbox(label='Cevap', lines=12)
    sources_box = gr.Textbox(label='Kullan?lan kaynaklar / retrieved chunks', lines=16)
    run_button.click(
        answer_question,
        inputs=[question_box, corpus_mode, generate_answer_box, llm_model_box, top_k_box, candidate_k_box, max_tokens_box],
        outputs=[answer_box, sources_box],
    )

demo.launch(share=True, debug=True)

## Demo Notes

- Use `official_law` for the final official-law system.
- Use `custom_docs` after running `21_custom_data_pipeline_demo.ipynb`.
- Keep `LLM cevab? ?ret` unchecked for fast source retrieval demos.
- Turn it on when you want an actual generated answer with citations.